# UCC static 5G dataset review

This notebook inspects the deterministic static-trace manifest. Selection and quality classification are implemented in the tested package.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
manifest = json.loads((root / 'manifests/ucc_static_v1.json').read_text())
manifest['inventory']

In [ ]:
rows = []
for trace in manifest['traces']:
    window = trace['selected_window'] or {}
    metrics = window.get('metrics', {})
    rows.append({
        'trace_id': trace['trace_id'],
        'application': trace['app'],
        'classification': trace['classification'],
        'window_start': window.get('start'),
        'coverage': window.get('timestamp_coverage'),
        'speed_p95_kph': window.get('speed_p95_kph'),
        'rsrp_p50_dbm': metrics.get('RSRP', {}).get('p50'),
        'rsrq_p50_db': metrics.get('RSRQ', {}).get('p50'),
        'snr_p50_db': metrics.get('SNR', {}).get('p50'),
        'flags': ', '.join(trace['quality_flags']),
    })
traces = pd.DataFrame(rows)
traces

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name, group in traces.groupby('classification'):
    ax.scatter(group['rsrp_p50_dbm'], group['snr_p50_db'], label=name, s=55)
ax.set_xlabel('Median RSRP (dBm)')
ax.set_ylabel('Median SNR (dB)')
ax.grid(alpha=0.25)
ax.legend();

In [ ]:
traces.loc[traces['flags'].ne(''), ['application', 'classification', 'speed_p95_kph', 'flags']]